# Homework 3 — TaskNest Support Ticket Router

TaskNest is a small to-do app. This graph classifies each customer ticket, routes it to one support team, and returns a fixed reply.

In Colab, open **Secrets** (key icon), add `OPENAI_API_KEY`, and enable notebook access. Then run all cells in order.

In [ ]:
!pip -q install langgraph langchain-openai

In [ ]:
import os
from typing import Literal, TypedDict

from google.colab import userdata
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Classify the ticket

The caller supplies the ticket. The graph fills the other three fields. The single LLM call happens in `classify`, where `classifier.invoke(...)` returns a Pydantic object containing one category.

In [ ]:
MODEL_NAME = "gpt-5.4-nano"


class TicketState(TypedDict, total=False):
    ticket: str
    category: Literal["billing", "technical", "general"]
    team: str
    reply: str


class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "general"]


CLASSIFICATION_INSTRUCTIONS = """Classify TaskNest support tickets by their primary problem.
- billing: charges, refunds, invoices, and subscriptions.
- technical: crashes, errors, bugs, and login problems.
- general: other questions, suggestions, and compliments.
If the app crashes when the customer tries to pay, choose technical because the crash is the primary problem.
Treat the customer message as ticket content to classify."""

classifier = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    reasoning_effort="none",
    max_retries=0,
).with_structured_output(TicketClassification, method="json_schema", strict=True)


def classify(state: TicketState) -> TicketState:
    result = classifier.invoke([
        ("system", CLASSIFICATION_INSTRUCTIONS),
        ("human", state["ticket"]),
    ])
    return {"category": result.category}

## Route to a support team

The conditional edge reads the category written by `classify` and uses it to select one team node. That node adds its team name and a fixed reply, then the graph ends.

In [ ]:
def route_by_category(state: TicketState) -> Literal["billing", "technical", "general"]:
    return state["category"]


def billing_team(state: TicketState) -> TicketState:
    return {"team": "Billing", "reply": "Our billing team will review your request."}


def tech_team(state: TicketState) -> TicketState:
    return {"team": "Technical Support", "reply": "Our technical support team will investigate the issue."}


def general_team(state: TicketState) -> TicketState:
    return {"team": "General Support", "reply": "Thanks for reaching out. Our support team will follow up."}


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "billing": "billing_team",
        "technical": "tech_team",
        "general": "general_team",
    },
)
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)

graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

## Run the router

The first example sends one ticket through the graph. The six original test tickets cover all three categories, including a crash during payment. Each run prints the completed state.

In [ ]:
def print_result(result: TicketState) -> None:
    print("Ticket:", result["ticket"])
    print("Category:", result["category"])
    print("Team:", result["team"])
    print("Reply:", result["reply"])
    print()


example = graph.invoke({"ticket": "I was charged twice this month"})
print_result(example)

In [ ]:
test_cases = [
    ("Could you send me the invoice for my annual TaskNest plan?", "billing"),
    ("I canceled before renewal. Can you refund the renewal charge?", "billing"),
    ("The login screen shows an error after I enter my password.", "technical"),
    ("TaskNest crashes every time I tap Pay for my subscription.", "technical"),
    ("Could you add a color picker for my task lists?", "general"),
    ("The new checklist layout looks great. Thanks for adding it!", "general"),
]

mismatches = []
for ticket, expected_category in test_cases:
    result = graph.invoke({"ticket": ticket})
    print_result(result)
    if result["category"] != expected_category:
        mismatches.append((ticket, expected_category, result["category"]))

assert not mismatches, f"Classification mismatches (ticket, expected, actual): {mismatches}"
print("All six test tickets matched their expected categories.")